# Create the Bell states with the CUDA-Q Kernel Builder


In [1]:
import cudaq

# Set up the CUDA-Q target for multiple QPUs if available
cudaq.set_target("nvidia", option="mqpu")
print(f"Number of QPUs available: {cudaq.get_target().num_qpus()}")

Number of QPUs available: 1


In [ ]:
num_qubits = 2 
num_iters = 100

def create_bell_state(num_qubits):
    # 1. Construct the kernel
    kernel = cudaq.make_kernel()
    q = kernel.qalloc(num_qubits)

    # Apply the Hadamard and CNOT gates
    kernel.h(q[0])
    kernel.cx(q[0], q[1]) # kernel.cx is the builder equivalent to x.ctrl

    # Measure the qubits (returns a QuakeValue handle, not a classical string)
    kernel.mz(q) 
    return kernel

# 2. Run the kernel and sample the results
# This handles the repetition (looping) and resets on the QPU/simulator
kernel = create_bell_state(num_qubits)
results = cudaq.sample(kernel, shots_count=num_iters)

# 3. Process the results to get the count of matching states
ncorrect = 0
for bitstring, count in results.items():
    # If the measurement of both qubits matches (e.g., '00' or '11')
    if bitstring[0] == bitstring[1]:
        ncorrect += count

print(f"N correct = {ncorrect}")
print(cudaq.draw(kernel))
    

N correct = 100
     ╭───╮     
q0 : ┤ h ├──●──
     ╰───╯╭─┴─╮
q1 : ─────┤ x ├
          ╰───╯

